# `process(StateChange)` — Inducing Dyslexia in Qwen2.5-VL

Demonstrates the `state_change_fn` mechanism using the Honarmand-style induced-dyslexia paradigm:

1. **Render stimuli**: real-word and pseudoword images (test + localizer)
2. **Localize**: identify late-LM-layer units that respond more to real words than pseudowords
3. **Lesion**: ablate those units via `process(StateChange(...))`
4. **Re-test** behaviorally: ask Qwen to classify each test image as `real` or `pseudo` via generation, before vs after the lesion
5. **Reset** and confirm restoration

**Why Qwen instead of CLIP?** VWFA proper is in left ventral occipitotemporal cortex (vision area), but the Honarmand et al. (2026) paper finds the dyslexia *behavioral* deficit emerges most clearly when you ablate **language-model-side** units in a VLM — the place where visual word forms link to phonology and meaning. CLIP has no LM decoder, so ablating its vision tower produces only a small effect. Qwen2.5-VL has a 36-layer causal LM downstream of the vision tower; lesioning its late layers is closer to the actual paper's setup.

**Hardware note**: Qwen2.5-VL-3B is a 3B-parameter VLM; this notebook needs a GPU (CUDA or MPS) to run in reasonable time. On EC2 g5.4xlarge it's ~3 min end-to-end.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

import brainscore
import brainscore_vision  # registers Qwen2.5-VL
from brainscore_core.model_interface import (
    StateChange, Selection, Perturbation,
)
from brainscore.perturbation import build_pytorch_ablation_fn


def render_word_image(word, size=224, font_size=56):
    """Render `word` as black text on a white 224x224 image."""
    img = Image.new('RGB', (size, size), color='white')
    draw = ImageDraw.Draw(img)
    for path in ['/System/Library/Fonts/Helvetica.ttc',
                 '/System/Library/Fonts/Supplemental/Arial.ttf',
                 '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf']:
        try:
            font = ImageFont.truetype(path, font_size); break
        except (OSError, IOError):
            continue
    else:
        font = ImageFont.load_default()
    bbox = draw.textbbox((0, 0), word, font=font)
    x = (size - (bbox[2] - bbox[0])) // 2 - bbox[0]
    y = (size - (bbox[3] - bbox[1])) // 2 - bbox[1]
    draw.text((x, y), word, fill='black', font=font)
    return img


## 1. Stimuli

Smaller stimulus sets than the CLIP version because each Qwen forward pass is ~10-100x slower:
- **Test set** (3 real + 3 pseudo): held-out for measuring the behavioral effect of the lesion.
- **Localizer** (6 real + 6 pseudo): used only to identify which late-LM units are word-selective.

In [ ]:
TEST_REAL    = ['apple',    'school',  'ocean']
TEST_PSEUDO  = ['blapt',    'pringol', 'mishel']

LOC_REAL     = ['mountain', 'forest', 'sunset', 'tiger', 'doctor', 'rocket']
LOC_PSEUDO   = ['plapp',    'snorel', 'graspol','mendrel','flarbon','tronkel']

test_imgs    = [render_word_image(w) for w in TEST_REAL + TEST_PSEUDO]
test_labels  = (['real:'   + w for w in TEST_REAL]
                + ['pseudo:' + w for w in TEST_PSEUDO])
test_truth   = ['real'] * len(TEST_REAL) + ['pseudo'] * len(TEST_PSEUDO)
loc_real_imgs   = [render_word_image(w) for w in LOC_REAL]
loc_pseudo_imgs = [render_word_image(w) for w in LOC_PSEUDO]

print(f'Test set:      {len(test_imgs)} images')
print(f'Localizer set: {len(loc_real_imgs) + len(loc_pseudo_imgs)} images')


### Display the stimuli

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(13, 6.5))

# Row 0 — test set
for i, (img, lbl) in enumerate(zip(test_imgs, test_labels)):
    axes[0, i].imshow(img)
    color = 'navy' if lbl.startswith('real') else 'darkred'
    axes[0, i].set_title(lbl, fontsize=11, color=color)
    axes[0, i].axis('off')
axes[0, 0].set_ylabel('TEST', fontsize=12, fontweight='bold',
                     rotation=0, ha='right', va='center')

# Row 1 — localizer real
for i, (img, w) in enumerate(zip(loc_real_imgs, LOC_REAL)):
    axes[1, i].imshow(img); axes[1, i].axis('off')
    axes[1, i].set_title(f'real:{w}', fontsize=10, color='navy')
axes[1, 0].set_ylabel('LOCALIZER\nreal', fontsize=11, fontweight='bold',
                     rotation=0, ha='right', va='center')

# Row 2 — localizer pseudo
for i, (img, w) in enumerate(zip(loc_pseudo_imgs, LOC_PSEUDO)):
    axes[2, i].imshow(img); axes[2, i].axis('off')
    axes[2, i].set_title(f'pseudo:{w}', fontsize=10, color='darkred')
axes[2, 0].set_ylabel('LOCALIZER\npseudo', fontsize=11, fontweight='bold',
                     rotation=0, ha='right', va='center')

plt.tight_layout()
plt.show()


## 2. Load Qwen2.5-VL and wire `state_change_fn`

Pull Qwen via the brain-score registry. The registration already configures the vision wrapper, text wrapper, and a `generation_fn` for instruction-following classification. We attach a `state_change_fn` here in the notebook to make the wiring explicit.

Layer paths in `StateChange.target` resolve against the *full* `qwen_model` object (the `Qwen2_5_VLForConditionalGeneration` returned by `from_pretrained`). The path to the late LM layer is therefore `model.language_model.layers.28` — the 29th of 36 decoder layers, where `'language_system'` lives in Qwen's `region_layer_map`.

In [ ]:
bs_model = brainscore.load_model('qwen2.5-vl-3b')
qwen_model = bs_model._model
qwen_processor = bs_model._preprocessors['vision'].processor

device = next(qwen_model.parameters()).device
print(f'Model on device: {device}')

bs_model._state_change_fn = build_pytorch_ablation_fn(qwen_model)
print(f'state_change_fn wired: {bs_model._state_change_fn is not None}')


## 3. Localize word-selective units in the late language model

Run each localizer image through the full Qwen forward pass with a generic reading prompt (`"What word is shown in this image?"`). Capture activations at `model.language_model.layers.28`, take the mean over the sequence dimension to get per-unit activations, and compute selectivity = `mean(real activations) − mean(pseudo activations)`.

High positive selectivity → unit fires more for real words than pseudowords → candidate target for ablation.

In [ ]:
TARGET_LAYER_PATH = 'model.language_model.layers.28'
target_layer = qwen_model.model.language_model.layers[28]
captured = []

def _hook(_m, _i, output):
    h = output[0] if isinstance(output, tuple) else output
    captured.append(h.detach().to('cpu', dtype=torch.float32).numpy())

def encode_image(img, prompt='What word is shown in this image?'):
    """Forward pass an image+prompt through Qwen; capture hook fires."""
    messages = [{'role': 'user',
                 'content': [{'type': 'image', 'image': img},
                             {'type': 'text',  'text': prompt}]}]
    text = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_processor(text=[text], images=[img],
                            return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        qwen_model(**inputs)

captured.clear()
h = target_layer.register_forward_hook(_hook)
for im in loc_real_imgs:
    encode_image(im)
loc_real_act = np.stack([a.mean(axis=1).squeeze(0) for a in captured])
captured.clear()
for im in loc_pseudo_imgs:
    encode_image(im)
loc_pseudo_act = np.stack([a.mean(axis=1).squeeze(0) for a in captured])
h.remove()

selectivity = loc_real_act.mean(0) - loc_pseudo_act.mean(0)
TOP_K = 200  # ~6% of Qwen's 3072 LM hidden dim
vwfa_units = np.argsort(selectivity)[-TOP_K:].tolist()

print(f'Hidden dim:        {selectivity.shape[0]}')
print(f'Selectivity range: [{selectivity.min():+.4f}, {selectivity.max():+.4f}]')
print(f'Top-{TOP_K} word-selective units identified.')


## 4. Behavioral test — Qwen classifies each image as real / pseudo via generation

Ask Qwen the lexical-decision question: `"Is the word in this image a real English word? Answer 'real' or 'pseudo'."` and parse the response. This is exactly the ROAR paradigm (Honarmand et al. 2026) but on a small synthetic test set so it runs fast.

Run three times: baseline → after `process(StateChange(...))` lesion → after `model.reset()`.

In [ ]:
PROMPT = ("Is the word in this image a real English word? "
          "Answer with exactly one of: real, pseudo.")

def classify(img, max_new_tokens=4):
    messages = [{'role': 'user',
                 'content': [{'type': 'image', 'image': img},
                             {'type': 'text',  'text': PROMPT}]}]
    text = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_processor(text=[text], images=[img],
                            return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        out_ids = qwen_model.generate(**inputs,
                                       max_new_tokens=max_new_tokens,
                                       do_sample=False)
    response = qwen_processor.tokenizer.decode(
        out_ids[0, inputs['input_ids'].shape[1]:],
        skip_special_tokens=True).strip().lower()
    if 'real'   in response: return 'real'
    if 'pseudo' in response: return 'pseudo'
    return response  # uninterpretable

def acc(pred, truth):
    return float(np.mean([p == t for p, t in zip(pred, truth)]))

print('Baseline:')
baseline_pred = [classify(im) for im in test_imgs]
for lbl, pred in zip(test_labels, baseline_pred):
    mark = '✓' if pred == lbl.split(':')[0] else '✗'
    print(f'  {mark} {lbl:18s} → {pred}')
print(f'  baseline accuracy: {acc(baseline_pred, test_truth):.2f}')

applied = bs_model.process(StateChange(
    kind='ablation',
    target=Selection(layer=TARGET_LAYER_PATH, indices=vwfa_units),
    perturbation=Perturbation(kind='zero'),
))
print(f'\nLesion installed — handle_id: {applied.handle_id[:12]}...\n')
print('Ablated:')
ablated_pred = [classify(im) for im in test_imgs]
for lbl, pred in zip(test_labels, ablated_pred):
    mark = '✓' if pred == lbl.split(':')[0] else '✗'
    print(f'  {mark} {lbl:18s} → {pred}')
print(f'  ablated accuracy:  {acc(ablated_pred, test_truth):.2f}')

bs_model.reset()
print('\nRestored:')
restored_pred = [classify(im) for im in test_imgs]
for lbl, pred in zip(test_labels, restored_pred):
    mark = '✓' if pred == lbl.split(':')[0] else '✗'
    print(f'  {mark} {lbl:18s} → {pred}')
print(f'  restored accuracy: {acc(restored_pred, test_truth):.2f}')


## 5. Visualize

Two panels:
- **(left)** Per-unit word selectivity at `model.language_model.layers.28`. Red dashed line marks the ablation threshold; the units above it (top 200) get zeroed.
- **(right)** Per-stimulus classification — baseline / ablated / restored — colored green if correct, red if wrong. The induced-dyslexia signature: more red bars in the ablated column.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (left) selectivity histogram
axes[0].hist(selectivity, bins=80, color='steelblue', edgecolor='white')
thr = selectivity[vwfa_units].min()
axes[0].axvline(thr, color='red', ls='--', label=f'top {TOP_K} (ablated)')
axes[0].set_xlabel('mean(real) − mean(pseudo) activation')
axes[0].set_ylabel('# units')
axes[0].set_title('Per-unit word selectivity\n' + TARGET_LAYER_PATH)
axes[0].legend()

# (right) per-stimulus classification across the three conditions
conditions = ['baseline', 'ablated', 'restored']
preds = {'baseline': baseline_pred,
         'ablated':  ablated_pred,
         'restored': restored_pred}
y = np.arange(len(test_imgs))
for col, cond in enumerate(conditions):
    for row, (pred, truth, lbl) in enumerate(zip(preds[cond], test_truth, test_labels)):
        correct = (pred == truth)
        color = '#3aa55a' if correct else '#cc4444'
        axes[1].barh(row, 1, left=col, color=color, edgecolor='white')
        axes[1].text(col + 0.5, row, pred[:6], ha='center', va='center',
                     color='white', fontsize=9, fontweight='bold')
axes[1].set_yticks(y)
axes[1].set_yticklabels(test_labels, fontsize=9)
axes[1].set_xticks([0.5, 1.5, 2.5])
axes[1].set_xticklabels(['BASELINE',
                         f'ABLATED\n(induced dyslexia)',
                         'RESTORED'])
axes[1].set_xlim(0, 3)
axes[1].invert_yaxis()
axes[1].set_title('Generative classification (green = correct, red = wrong)')
axes[1].axhline(len(TEST_REAL) - 0.5, color='black', lw=0.5, ls=':')
for c in [1, 2]:
    axes[1].axvline(c, color='black', lw=0.5)

plt.tight_layout()
plt.show()

print(f'Accuracy: baseline={acc(baseline_pred, test_truth):.2f}  '
      f'ablated={acc(ablated_pred, test_truth):.2f}  '
      f'restored={acc(restored_pred, test_truth):.2f}')


## What this enables

- **Causal contribution measurement**: `(baseline accuracy) − (ablated accuracy)` is the causal contribution of those localizer-identified late-LM units to lexical decision in Qwen.
- **Honarmand et al. (2026) replication**: the experimental design here matches the paper's induced-dyslexia paradigm; swap our 6-image test set for the actual ROAR test stimuli (`Yeatman2021-lexical_decision-image` benchmark) for a published-comparable result.
- **Cross-model comparison**: identical code works on BLIP-2 by changing `'qwen2.5-vl-3b'` → `'blip2-opt-2.7b'` and adjusting `TARGET_LAYER_PATH` to the equivalent late decoder layer (`language_model.model.decoder.layers.28`).
- **Selective undo**: `process(StateChange(kind='reset', handle_id=...))` removes one perturbation at a time; `model.reset()` clears all. Multi-region lesion experiments compose cleanly without state leaking between conditions.

All from a single `BrainScoreModel` registration — no benchmark-side branching, no per-experiment subclassing.